# ELM Clusters vs Fixed Dataset Labels

This notebook analyzes how `elm_motif_cluster` assignments from
`data/derived/elm/elm_motif_clustering.csv.gz` distribute across fixed
dataset labels in `data/derived/fixed/cluster_fixed_labels.csv.gz`.

Questions:
- A. Which ELM clusters are only present in proteins without fixed labels?
- B. Which ELM clusters are only present in the test set?
- C. Which ELM clusters are only present in the training set?
- D. Which ELM clusters are present in both test and training sets?

In this repository, fixed label `caid` is treated as the test set.

In [6]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using CSV
using DataFrames

  Activating project at `~/.julia/dev/IDRBindingDataset.jl/scripts`


In [7]:
repo_root = normpath(joinpath(@__DIR__, ".."))
elm_path = joinpath(repo_root, "data", "derived", "elm", "elm_motif_clustering.csv.gz")
fixed_path = joinpath(repo_root, "data", "derived", "fixed", "cluster_fixed_labels.csv.gz")

elm = CSV.read(elm_path, DataFrame)
fixed = CSV.read(fixed_path, DataFrame)

fixed_labels = select(fixed, :protein_id, :cluster_fixed_dataset_label)
elm_fixed = leftjoin(elm, fixed_labels, on = :protein_id)

println("ELM assignments: ", nrow(elm))
println("ELM proteins: ", length(unique(elm.protein_id)))
println("ELM clusters: ", length(unique(elm.elm_motif_cluster)))

unmatched = setdiff(unique(elm.protein_id), unique(fixed_labels.protein_id))
println("ELM proteins missing in fixed table: ", length(unmatched))

ELM assignments: 462
ELM proteins: 442
ELM clusters: 10
ELM proteins missing in fixed table: 0


In [8]:
function normalize_fixed_group(label)
    if ismissing(label)
        return "without_fixed_label"
    end
    label_lower = lowercase(String(label))
    if label_lower in ("caid", "test", "testing")
        return "test"
    elseif label_lower == "training"
        return "training"
    else
        return String(label)
    end
end

elm_fixed.fixed_group = normalize_fixed_group.(elm_fixed.cluster_fixed_dataset_label)
first(elm_fixed, 5)

Row,protein_id,elm_motif_cluster,cluster_fixed_dataset_label,fixed_group
,String15,Int64,String15?,String
1,B7UM99,1,caid,test
2,D0NBE6,1,training,training
3,O00268,1,caid,test
4,O00418,1,training,training
5,O00499,1,caid,test


In [9]:
distribution = combine(
    groupby(elm_fixed, [:elm_motif_cluster, :fixed_group]),
    nrow => :n_protein_assignments,
)
cluster_totals = combine(groupby(elm_fixed, :elm_motif_cluster), nrow => :cluster_total)
distribution = leftjoin(distribution, cluster_totals, on = :elm_motif_cluster)
distribution.pct_in_cluster = round.(100 .* distribution.n_protein_assignments ./ distribution.cluster_total; digits = 2)
sort!(distribution, [:elm_motif_cluster, :fixed_group])

distribution

Row,elm_motif_cluster,fixed_group,n_protein_assignments,cluster_total,pct_in_cluster
,Int64,String,Int64,Int64?,Float64
1,1,test,65,433,15.01
2,1,training,343,433,79.21
3,1,without_fixed_label,25,433,5.77
4,2,training,3,3,100.0
5,3,test,1,4,25.0
6,3,training,2,4,50.0
7,3,without_fixed_label,1,4,25.0
8,4,training,3,3,100.0
9,5,training,1,1,100.0


In [10]:
function classify_elm_cluster(groups)
    unique_groups = Set(groups)
    if unique_groups == Set(["without_fixed_label"])
        return "A_without_fixed_labels_only"
    elseif unique_groups == Set(["test"])
        return "B_test_set_only"
    elseif unique_groups == Set(["training"])
        return "C_training_set_only"
    else
        return "mixed"
    end
end

cluster_presence = combine(
    groupby(elm_fixed, :elm_motif_cluster),
    :fixed_group => (groups -> "without_fixed_label" in groups) => :has_without_fixed_label,
    :fixed_group => (groups -> "test" in groups) => :has_test,
    :fixed_group => (groups -> "training" in groups) => :has_training,
    :fixed_group => classify_elm_cluster => :exclusive_category,
    nrow => :n_protein_assignments,
)
sort!(cluster_presence, :elm_motif_cluster)

a_clusters = cluster_presence.elm_motif_cluster[cluster_presence.exclusive_category .== "A_without_fixed_labels_only"]
b_clusters = cluster_presence.elm_motif_cluster[cluster_presence.exclusive_category .== "B_test_set_only"]
c_clusters = cluster_presence.elm_motif_cluster[cluster_presence.exclusive_category .== "C_training_set_only"]
d_clusters = cluster_presence.elm_motif_cluster[cluster_presence.has_test .& cluster_presence.has_training]

println("A. only without fixed labels: ", isempty(a_clusters) ? "none" : join(a_clusters, ", "))
println("B. only test set: ", isempty(b_clusters) ? "none" : join(b_clusters, ", "))
println("C. only training set: ", isempty(c_clusters) ? "none" : join(c_clusters, ", "))
println("D. in both test and training sets: ", isempty(d_clusters) ? "none" : join(d_clusters, ", "))

cluster_presence

A. only without fixed labels: none
B. only test set: 10
C. only training set: 2, 4, 5, 6, 9
D. in both test and training sets: 1, 3, 7, 8


Row,elm_motif_cluster,has_without_fixed_label,has_test,has_training,exclusive_category,n_protein_assignments
,Int64,Bool,Bool,Bool,String,Int64
1,1,true,true,true,mixed,433
2,2,false,false,true,C_training_set_only,3
3,3,true,true,true,mixed,4
4,4,false,false,true,C_training_set_only,3
5,5,false,false,true,C_training_set_only,1
6,6,false,false,true,C_training_set_only,1
7,7,false,true,true,mixed,12
8,8,false,true,true,mixed,3
9,9,false,false,true,C_training_set_only,1
